# Gold Seller Dimension

This notebook builds the `dim_sellers` Gold model from the cleaned Silver sellers dataset.

**Grain:** One row per `seller_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_SELLERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/sellers"
)

GOLD_DIM_SELLERS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/dim_sellers"
)

print(f"Silver source: {SILVER_SELLERS_PATH}")
print(f"Gold target: {GOLD_DIM_SELLERS_PATH}")

## 2. Read Silver sellers

In [0]:
silver_sellers_df = (
    spark.read
    .format("delta")
    .load(SILVER_SELLERS_PATH)
)

silver_seller_count = silver_sellers_df.count()

print(f"Silver seller rows: {silver_seller_count:,}")

display(silver_sellers_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_sellers_df.columns)

if missing_columns:
    raise ValueError(
        f"Silver sellers is missing required columns: {sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Validate seller state values

In [0]:
valid_brazil_states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

unexpected_states_df = (
    silver_sellers_df
    .filter(
        F.col("seller_state").isNull()
        | ~F.col("seller_state").isin(valid_brazil_states)
    )
    .select("seller_state")
    .distinct()
)

unexpected_state_count = unexpected_states_df.count()

if unexpected_state_count > 0:
    display(unexpected_states_df)
    raise ValueError(
        f"Found {unexpected_state_count} unexpected or null seller state values."
    )

print("Seller state validation passed.")

## 5. Build seller dimension

In [0]:
dim_sellers_df = (
    silver_sellers_df
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "_silver_processed_at",
    )
    .withColumn(
        "seller_region",
        F.when(F.col("seller_state").isin("AC", "AP", "AM", "PA", "RO", "RR", "TO"), "North")
        .when(F.col("seller_state").isin("AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"), "Northeast")
        .when(F.col("seller_state").isin("DF", "GO", "MT", "MS"), "Central-West")
        .when(F.col("seller_state").isin("ES", "MG", "RJ", "SP"), "Southeast")
        .when(F.col("seller_state").isin("PR", "RS", "SC"), "South")
        .otherwise("Unknown")
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(dim_sellers_df.limit(10))

## 6. Validate seller dimension

In [0]:
dim_seller_count = dim_sellers_df.count()

duplicate_seller_count = (
    dim_sellers_df
    .groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_seller_id_count = (
    dim_sellers_df
    .filter(F.col("seller_id").isNull())
    .count()
)

if dim_seller_count == 0:
    raise ValueError("Seller dimension is empty.")

if dim_seller_count != silver_seller_count:
    raise ValueError(
        "Seller dimension row count does not match Silver sellers. "
        f"Silver: {silver_seller_count:,}, Gold: {dim_seller_count:,}"
    )

if duplicate_seller_count > 0:
    raise ValueError(
        f"Seller dimension contains {duplicate_seller_count:,} duplicate seller IDs."
    )

if null_seller_id_count > 0:
    raise ValueError(
        f"Seller dimension contains {null_seller_id_count:,} null seller IDs."
    )

print(f"Seller dimension rows: {dim_seller_count:,}")
print("Seller dimension grain validation passed.")

## 7. Write seller dimension to Gold

In [0]:
(
    dim_sellers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_SELLERS_PATH)
)

print(f"Seller dimension written to: {GOLD_DIM_SELLERS_PATH}")

## 8. Validate Gold output

In [0]:
written_dim_sellers_df = (
    spark.read
    .format("delta")
    .load(GOLD_DIM_SELLERS_PATH)
)

written_seller_count = written_dim_sellers_df.count()

if written_seller_count != dim_seller_count:
    raise ValueError(
        "Gold seller dimension write validation failed. "
        f"Expected: {dim_seller_count:,}, Written: {written_seller_count:,}"
    )

print(f"Written seller dimension rows: {written_seller_count:,}")
print("Gold seller dimension write validation passed.")

## 9. Inspect Gold seller dimension

In [0]:
written_dim_sellers_df.printSchema()

display(
    written_dim_sellers_df
    .orderBy("seller_id")
    .limit(10)
)